# Setup

## 1. Environment Setup

We use [`uv`](https://github.com/astral-sh/uv) for fast, reproducible package management.

The steps below:
1. Install `uv` into `~/.local/bin`
2. Create a virtual environment at `.venv/`
3. Install all required packages (This might take a while)

> **After running this cell, restart the kernel** so that the newly installed packages (especially `vllm` and `transformers`) are picked up by the current Python session.

### IN Colab

In [ ]:
!ls -lh /content/repo.zip
!rm -rf /content/CSE151B_Kaggle
!unzip -oq /content/repo.zip -d /content/CSE151B_Kaggle
%cd /content/CSE151B_Kaggle
!pwd
!ls

In [ ]:
!pip install prettyprint sympy numpy pandas matplotlib transformers accelerate vllm tqdm bitsandbytes ipykernel jupyter nvidia-nvjitlink antlr4-python3-runtime==4.11.1

In [ ]:
import os

# Point Colab's environment to the newly installed nvidia-nvjitlink package
lib_path = "/usr/local/lib/python3.12/dist-packages/nvidia/nvjitlink/lib"

if "LD_LIBRARY_PATH" in os.environ:
    os.environ["LD_LIBRARY_PATH"] += f":{lib_path}"
else:
    os.environ["LD_LIBRARY_PATH"] = lib_path

In [ ]:
# Optional: download the current Colab workspace archive.
RUN_WORKSPACE_DOWNLOAD = False

if RUN_WORKSPACE_DOWNLOAD:
    !zip -qr /content/CSE151B_Kaggle_colab_outputs.zip /content/CSE151B_Kaggle
    from google.colab import files
    files.download("/content/CSE151B_Kaggle_colab_outputs.zip")
else:
    print("Skipping workspace download.")


### Only if in DataHub

In [ ]:
# Install uv
!wget -qO- https://astral.sh/uv/install.sh | sh

!export PATH="/home/ugheewala/.local/bin:$PATH"

# Create a virtual environment
!/home/ugheewala/.local/bin/uv venv .venv --seed --clear

# Install dependencies — this is fast thanks to uv's parallel resolver
!.venv/bin/python -m pip install prettyprint sympy numpy pandas matplotlib transformers accelerate vllm tqdm bitsandbytes antlr4-python3-runtime==4.11.1 ipykernel jupyter

# Install Jupyter Kernel
!.venv/bin/python -m ipykernel install --user --name cse151b --display-name "Python (cse151b)"

print("Done. Restart the kernel before proceeding.")
print("Selection process: on top right, click on current kernel '(ususally named python)' -> 'select another kernel' -> 'Jupyter Kernel' -> 'Python (cse151b)'.")

In [ ]:
# !/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
#     "numpy<2" \
#     "torch==2.1.2+cu118" \
#     "transformers==4.51.3" \
#     "accelerate==0.34.2" \
#     "huggingface_hub>=0.23.0" \
#     "safetensors" \
#     "sentencepiece" \
#     "tqdm" \
#     "pandas" \
#     "matplotlib" \
#     "sympy" \
#     "antlr4-python3-runtime==4.11.1" \
#     --extra-index-url https://download.pytorch.org/whl/cu118

In [ ]:
# !/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
#     "nvidia-cusparse-cu11" \
#     "nvidia-cublas-cu11" \
#     "nvidia-cuda-runtime-cu11" \
#     "nvidia-cudnn-cu11"

In [ ]:
# activate venv after installation. This needs to be run everytime.
!source ./.venv/bin/activate

In [ ]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "numpy<2" \
    "torch==2.3.1+cu121" \
    "torchvision==0.18.1+cu121" \
    "torchaudio==2.3.1+cu121" \
    --extra-index-url https://download.pytorch.org/whl/cu121

In [ ]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "transformers==4.51.3" \
    "accelerate>=0.30.0" \
    "huggingface_hub>=0.23.0" \
    "safetensors" \
    "sentencepiece" \
    "tokenizers==0.21.4" \
    "sympy" \
    "pandas" \
    "matplotlib" \
    "tqdm" \
    "prettyprint" \
    "antlr4-python3-runtime==4.11.1" \
    "ipykernel" \
    "jupyter"

In [ ]:
!/home/<name>/.local/bin/uv pip install --python .venv/bin/python \
    "bitsandbytes==0.45.5"

## 2. Imports & Configuration

All key settings are collected in one place.  
- `DATA_PATH` — public dataset with ground-truth answers (use this to measure accuracy)
- `OUTPUT_PATH` — where per-question results will be written
- `GPU_ID` — which GPU to use (update if your machine has a different device index)
- `MAX_TOKENS` — maximum tokens the model may generate per response

In [ ]:
import os
import sys
import json
import time
import csv
import subprocess
from pathlib import Path
from pprint import pprint

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"                    # CUDA_VISIBLE_DEVICES
PUBLIC_DATA_PATH   = "data/public.jsonl"
PRIVATE_DATA_PATH   = "data/private.jsonl"
OUTPUT_PATH = "results/starter_results.jsonl"

PROJECT_ROOT = Path.cwd()

RESULTS_DIR = PROJECT_ROOT / "results"
BASELINE1_DIR = RESULTS_DIR / "baseline1_weakest"
BASELINE1_DIR.mkdir(parents=True, exist_ok=True)

VAL_FRAC = 0.20
SPLIT_SEED = 414

CACHE_DIR = None
HF_HOME_DIR = None

MAX_INPUT_TOKENS = 4096 #4096
MAX_MODEL_LEN = 4096 #4096

MAX_NEW_TOKENS_SMOKE = 256 #256
MAX_NEW_TOKENS_BASELINE = 1024 #1024

INFERENCE_BACKEND = "transformers"

BATCH_SIZE = 1
LOAD_IN_4BIT = True


MAX_TOKENS = 512

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

if HF_HOME_DIR is not None:
    os.environ["HF_HOME"] = str(HF_HOME_DIR)

if CACHE_DIR is not None:
    Path(CACHE_DIR).mkdir(parents=True, exist_ok=True)

print("HF_HOME      :", os.environ.get("HF_HOME"))
print("HF_HUB_CACHE :", os.environ.get("HF_HUB_CACHE"))
print("cache_dir    :", CACHE_DIR)

In [ ]:
import torch

print(f"CUDA_VISIBLE_DEVICES (Env): {os.environ.get('CUDA_VISIBLE_DEVICES')}")

cuda_available = torch.cuda.is_available()
print(f"Is CUDA available? {cuda_available}")

if cuda_available:
    print(f"Current Device: {torch.cuda.current_device()}")
    print(f"Device Name: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("PyTorch still can't see the GPU.")
    device = torch.device("cpu")

In [ ]:
# import site

# roots = [Path(p) for p in site.getsitepackages()]
# matches = []

# for root in roots:
#     if root.exists():
#         matches.extend(root.rglob("libcusparse.so*"))

# for m in matches:
#     print(m)

In [ ]:
# wanted_libs = {
#     "libcusparse.so",
#     "libcublas.so",
#     "libcudart.so",
#     "libcudnn.so",
# }

# lib_dirs = []

# for root in map(Path, site.getsitepackages()):
#     if not root.exists():
#         continue

#     for lib in wanted_libs:
#         for match in root.rglob(lib + "*"):
#             lib_dir = str(match.parent)
#             if lib_dir not in lib_dirs:
#                 lib_dirs.append(lib_dir)

# VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
# SITE = VENV / "lib/python3.11/site-packages"

# cuda11_dirs = [
#     SITE / "nvidia/cusparse/lib",
#     SITE / "nvidia/cublas/lib",
#     SITE / "nvidia/cuda_runtime/lib",
#     SITE / "nvidia/cudnn/lib",
#     SITE / "torch/lib",
# ]

# cuda11_dirs = [str(p) for p in cuda11_dirs if p.exists()]

# path_line = ":".join(cuda11_dirs)

# print("Add this before starting the notebook/kernel:")
# print(f'export LD_LIBRARY_PATH="{path_line}:$LD_LIBRARY_PATH"')

In [ ]:
# import os
# import subprocess
# from pathlib import Path

# VENV = Path("/home/ugheewala/private/CSE151B_Kaggle/.venv")
# SITE = VENV / "lib/python3.11/site-packages"

# cuda11_dirs = [
#     SITE / "nvidia/cusparse/lib",
#     SITE / "nvidia/cublas/lib",
#     SITE / "nvidia/cuda_runtime/lib",
#     SITE / "nvidia/cudnn/lib",
#     SITE / "torch/lib",
# ]

# env = os.environ.copy()
# env["LD_LIBRARY_PATH"] = ":".join(str(p) for p in cuda11_dirs if p.exists()) + ":" + env.get("LD_LIBRARY_PATH", "")

# subprocess.run(
#     [str(VENV / "bin/python"), "-m", "bitsandbytes"],
#     env=env,
# )

In [ ]:
# import json
# from pathlib import Path

# kernel_json = Path("/home/ugheewala/.local/share/jupyter/kernels/cse151b/kernel.json")

# with open(kernel_json, "r") as f:
#     spec = json.load(f)

# ld_library_path = (
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cusparse/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cublas/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cuda_runtime/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/nvidia/cudnn/lib:"
#     "/home/ugheewala/private/CSE151B_Kaggle/.venv/lib/python3.11/site-packages/torch/lib:"
#     "${LD_LIBRARY_PATH}"
# )

# spec.setdefault("env", {})
# spec["env"]["LD_LIBRARY_PATH"] = ld_library_path
# spec["env"]["BNB_CUDA_VERSION"] = "118"

# with open(kernel_json, "w") as f:
#     json.dump(spec, f, indent=2)

# print(kernel_json)
# print(json.dumps(spec, indent=2))

In [ ]:
import transformers

print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("cuda available:", torch.cuda.is_available())
print("transformers:", transformers.__version__)

In [ ]:
try:
    import bitsandbytes as bnb
    print("bitsandbytes:", bnb.__version__)
except Exception as e:
    print("bitsandbytes import failed:", repr(e))

In [ ]:
from transformers.utils import is_torch_available, is_bitsandbytes_available

print("is_torch_available:", is_torch_available())
print("is_bitsandbytes_available:", is_bitsandbytes_available())

In [ ]:
from transformers import AutoTokenizer
from tqdm import tqdm

from baseline.datasets import load_public_splits, load_private_set
from baseline.generation import GenerationConfig
from baseline.prompt_sets import build_prompt_texts
from baseline.modeling import ModelConfig, load_transformers_model, predownload_model, load_model, detect_gpu_info
from baseline.scoring import load_judger, score_one, summarize_results
from baseline.progress_viz import RunProgressDashboard
from prompting.prompt_chain import build_prompt_chain
from baseline.runner import run_problem_set, benchmark_batch_sizes, write_report

In [ ]:
gpu_info = detect_gpu_info()
pprint(gpu_info.to_dict())

## 3. Load the Dataset

The dataset is stored as newline-delimited JSON (`.jsonl`). Each line is one question with the following fields:

| Field | Description |
|---|---|
| `id` | Unique question identifier |
| `question` | Problem statement |
| `options` | List of answer choices - present for **MCQ**, absent for **free-form** |
| `answer` | Ground-truth answer (letter for MCQ, value/list for free-form) |

In [ ]:
splits = load_public_splits(PUBLIC_DATA_PATH, val_frac=VAL_FRAC, seed=SPLIT_SEED)

train_set = splits["train"]
val_set = splits["val"]
public_set = splits["public"]
private_set = load_private_set(PRIVATE_DATA_PATH)

print("Train summary:")
pprint(train_set.summary())

print("\nValidation summary:")
pprint(val_set.summary())

print("\nPublic summary:")
pprint(public_set.summary())

print("\nPrivate summary:")
pprint(private_set.summary())

In [ ]:
prompt_chain = build_prompt_chain(strategy_name="baseline")

for label, problem_set in [("train", train_set), ("val", val_set), ("private", private_set)]:
    problem = problem_set.problems()[0]
    spec = prompt_chain.build_spec(problem)

    print("=" * 80)
    print(label, "id=", problem.id, "template=", spec.name)
    print("metadata:", spec.metadata)
    print("generation_hints:", spec.generation_hints)
    print(spec.to_messages()[0]["content"][:300])
    print("--- user ---")
    print(spec.to_messages()[-1]["content"][:500])

## 4. Modeling

In [ ]:
model_config = ModelConfig(
    model_id=MODEL_ID,
    backend=INFERENCE_BACKEND,
    cache_dir=CACHE_DIR,
    gpu_id=GPU_ID,
    max_input_tokens=MAX_INPUT_TOKENS,
    max_model_len=MAX_MODEL_LEN,
    dtype="bfloat16",
    torch_dtype="bfloat16",
    load_in_4bit=True,
    device_map="auto",
    low_cpu_mem_usage=True,
    gpu_memory_utilization=0.85,
    max_num_seqs=256,
    max_num_batched_tokens=MAX_TOKENS,
    reuse_loaded=True,
)

t0 = time.perf_counter()
model_bundle = load_model(model_config)
print(f"Model load/reuse time: {time.perf_counter() - t0:.2f} sec")
print("Backend:", model_bundle.backend)
print("Device:", model_bundle.device())

# Baseline 1: Naive

In [ ]:
baseline1_prompt_chain = build_prompt_chain(strategy_name="baseline")

smoke_generation_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS_SMOKE,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

baseline1_generation_config = GenerationConfig(
    max_new_tokens=MAX_NEW_TOKENS_BASELINE,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

In [ ]:
batch_report = benchmark_batch_sizes(
    problem_set=train_set,
    model_bundle=model_bundle,
    batch_sizes=[1, 2, 4, 8, 16, 32, 64],
    prompt_chain=baseline1_prompt_chain,
    generation_config=smoke_generation_config,
    sample_size=16,
    score=False,
    show_progress=False,
)

pd.DataFrame(batch_report["rows"])

In [ ]:
# Train split smoke run

baseline1_train_result = run_problem_set(
    problem_set=train_set,
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=32,
    score=True,
    output_jsonl_path=BASELINE1_DIR / "train_results.jsonl",
    report_json_path=BASELINE1_DIR / "train_report.json",
)

pprint(baseline1_train_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.3076923076923077,
             'mcq_acc': 0.05263157894736842,
             'n_correct': 5,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.15625},
 'timings': {'generation_sec': 11.5697886199996,
             'prompt_build_sec': 0.002926810000644764,
             'scoring_sec': 0.7455679300001066}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.6923076923076923,
             'mcq_acc': 0.7368421052631579,
             'n_correct': 23,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.71875},
 'timings': {'generation_sec': 461.65091967599983,
             'prompt_build_sec': 0.0029734310001003905,
             'scoring_sec': 1.0903050570000232}}
"""

In [ ]:
# Validation run

baseline1_val_result = run_problem_set(
    problem_set=val_set,
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=True,
    output_jsonl_path=BASELINE1_DIR / "val_results.jsonl",
    report_json_path=BASELINE1_DIR / "val_report.json",
)

pprint(baseline1_val_result.report)

In [ ]:
"""
 'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.12666666666666668,
             'mcq_acc': 0.08,
             'n_correct': 25,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.1111111111111111},
 'timings': {'generation_sec': 87.61873525499959,
             'prompt_build_sec': 0.01957373799996276,
             'scoring_sec': 7.864173332000064}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.54,
             'mcq_acc': 0.8266666666666667,
             'n_correct': 143,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.6355555555555555},
 'timings': {'generation_sec': 2392.0188707439997,
             'prompt_build_sec': 0.038322351000260824,
             'scoring_sec': 16.22751727099967}}
"""

In [ ]:
# Test run

PRIVATE_LIMIT = None

baseline1_private_result = run_problem_set(
    problem_set=private_set,
    model_bundle=model_bundle,
    prompt_chain=baseline1_prompt_chain,
    generation_config=baseline1_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=False,
    output_jsonl_path=BASELINE1_DIR / "private_results.jsonl",
    submission_csv_path=BASELINE1_DIR / "submission.csv",
    report_json_path=BASELINE1_DIR / "private_report.json",
)

pprint(baseline1_private_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/results/baseline1_weakest/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 339.3967333410001,
             'prompt_build_sec': 0.07438255999932153,
             'scoring_sec': 0.0008840780001264648}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 12278.958291005998,
             'prompt_build_sec': 0.07650407100027223,
             'scoring_sec': 0.002097485998092452}}
"""

In [ ]:
# At the end to download shit

!zip -r B1_results.zip /content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline1_weakest

from google.colab import files
files.download('B1_results.zip')


In [ ]:
submission_path = BASELINE1_DIR / "submission.csv"

if submission_path.exists():
    sub_df = pd.read_csv(submission_path)
    print(sub_df.shape)
    display(sub_df.head())
    print("Columns:", list(sub_df.columns))
else:
    print("No submission file yet. Run the private test cell first.")

# Baseline 2: Output hardening

In [ ]:
from baseline.baseline2_runner import run_baseline2_problem_set
from baseline.generation import GenerationConfig

BASELINE2_DIR = RESULTS_DIR / "baseline2_prompt_format"
BASELINE2_DIR.mkdir(parents=True, exist_ok=True)

baseline2_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,#4096,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

baseline2_retry_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,#1024,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=False,
)

In [ ]:
print("Model backend:", model_bundle.backend)
print("Model max_model_len:", model_bundle.config.max_model_len)
print("Generation max_new_tokens:", baseline2_generation_config.max_new_tokens)
print("Retry max_new_tokens:", baseline2_retry_generation_config.max_new_tokens)

from baseline.prompt_sets import build_prompt_texts
from prompting.prompt_chain import build_prompt_chain

prompt_chain = build_prompt_chain(strategy_name="baseline2")
prompt_rows = build_prompt_texts(val_set.head(5), model_bundle.tokenizer, prompt_chain=prompt_chain)

for row in prompt_rows:
    toks = model_bundle.tokenizer.encode(row["prompt_text"])
    print(row["id"], "prompt_tokens:", len(toks))

for row in prompt_rows:
  prompt_tokens = len(model_bundle.tokenizer.encode(row["prompt_text"]))
  available = model_bundle.config.max_model_len - prompt_tokens
  effective_new_token_cap = min(baseline2_generation_config.max_new_tokens, available)

  print({
      "id": row["id"],
      "prompt_tokens": prompt_tokens,
      "max_model_len": model_bundle.config.max_model_len,
      "configured_max_new_tokens": baseline2_generation_config.max_new_tokens,
      "effective_new_token_cap": effective_new_token_cap,
  })

In [ ]:
baseline2_train_result = run_baseline2_problem_set(
    problem_set=train_set,
    model_bundle=model_bundle,
    generation_config=baseline2_generation_config,
    retry_generation_config=baseline2_retry_generation_config,
    batch_size=BATCH_SIZE,
    limit=32,
    score=True,
    output_jsonl_path=BASELINE2_DIR / "train_results.jsonl",
    debug_jsonl_path=BASELINE2_DIR / "train_debug.jsonl",
    report_json_path=BASELINE2_DIR / "train_report.json",
    show_progress=True,
)

pprint(baseline2_train_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'schema_accuracy': {'retry_used_accuracy': 0.6,
                     'retry_used_n': 5,
                     'schema_invalid_accuracy': 0.08333333333333333,
                     'schema_invalid_n': 24,
                     'schema_valid_accuracy': 0.875,
                     'schema_valid_n': 8},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.5384615384615384,
             'mcq_acc': 0.10526315789473684,
             'n_correct': 9,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.28125},
 'timings': {'generation_sec': 11.575837816000103,
             'prompt_build_sec': 0.023429658999930325,
             'retry_generation_sec': 11.110912010999982,
             'scoring_sec': 0.8481839019998461}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 4096,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 1024,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': 0.0,
                     'retry_used_n': 1,
                     'sanitized_accuracy': 0.6,
                     'sanitized_n': 5,
                     'schema_invalid_accuracy': 0.0,
                     'schema_invalid_n': 7,
                     'schema_valid_accuracy': 0.56,
                     'schema_valid_n': 25},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.46153846153846156,
             'mcq_acc': 0.42105263157894735,
             'n_correct': 14,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.4375},
 'timings': {'generation_sec': 46.100986020000164,
             'prompt_build_sec': 0.0028652999999394524,
             'retry_generation_sec': 8.265795843999967,
             'scoring_sec': 1.19583538400002}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/train_results.jsonl',
 'problem_set': {'n': 32,
                 'n_answered': 32,
                 'n_free_form': 13,
                 'n_mcq': 19,
                 'name': 'public_train_head32'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 32768,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': None,
                     'retry_used_n': 0,
                     'sanitized_accuracy': None,
                     'sanitized_n': 0,
                     'schema_invalid_accuracy': 0.0,
                     'schema_invalid_n': 2,
                     'schema_valid_accuracy': 0.7,
                     'schema_valid_n': 30},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.6923076923076923,
             'mcq_acc': 0.631578947368421,
             'n_correct': 21,
             'n_free_form': 13,
             'n_mcq': 19,
             'n_scored': 32,
             'overall_acc': 0.65625},
 'timings': {'generation_sec': 165.0723143110008,
             'prompt_build_sec': 0.0030510440010402817,
             'retry_generation_sec': 0.5391863780023414,
             'scoring_sec': 0.7430123269987234}}
"""

In [ ]:
baseline2_val_result = run_baseline2_problem_set(
    problem_set=val_set,
    model_bundle=model_bundle,
    generation_config=baseline2_generation_config,
    retry_generation_config=baseline2_retry_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=True,
    output_jsonl_path=BASELINE2_DIR / "val_results.jsonl",
    debug_jsonl_path=BASELINE2_DIR / "val_debug.jsonl",
    report_json_path=BASELINE2_DIR / "val_report.json",
    show_progress=True,
)

pprint(baseline2_val_result.report)

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'schema_accuracy': {'retry_used_accuracy': 0.6590909090909091,
                     'retry_used_n': 44,
                     'schema_invalid_accuracy': 0.17333333333333334,
                     'schema_invalid_n': 150,
                     'schema_valid_accuracy': 0.76,
                     'schema_valid_n': 75},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.4666666666666667,
             'mcq_acc': 0.17333333333333334,
             'n_correct': 83,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.3688888888888889},
 'timings': {'generation_sec': 82.84365255199987,
             'prompt_build_sec': 0.02054123199991409,
             'retry_generation_sec': 69.44308633399987,
             'scoring_sec': 14.325112181999884}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 4096,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 1024,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': 0.25,
                     'retry_used_n': 8,
                     'sanitized_accuracy': 0.3142857142857143,
                     'sanitized_n': 35,
                     'schema_invalid_accuracy': 0.05405405405405406,
                     'schema_invalid_n': 37,
                     'schema_valid_accuracy': 0.6063829787234043,
                     'schema_valid_n': 188},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.56,
             'mcq_acc': 0.4266666666666667,
             'n_correct': 116,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.5155555555555555},
 'timings': {'generation_sec': 341.6052394200001,
             'prompt_build_sec': 0.023227069000085976,
             'retry_generation_sec': 21.939288180000176,
             'scoring_sec': 19.803487271999984}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/val_results.jsonl',
 'problem_set': {'n': 225,
                 'n_answered': 225,
                 'n_free_form': 150,
                 'n_mcq': 75,
                 'name': 'public_val'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 32768,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': 1.0,
                     'retry_used_n': 2,
                     'sanitized_accuracy': 1.0,
                     'sanitized_n': 3,
                     'schema_invalid_accuracy': 0.07692307692307693,
                     'schema_invalid_n': 13,
                     'schema_valid_accuracy': 0.660377358490566,
                     'schema_valid_n': 212},
 'score_available': True,
 'submission_csv_path': None,
 'summary': {'free_form_acc': 0.56,
             'mcq_acc': 0.76,
             'n_correct': 141,
             'n_free_form': 150,
             'n_mcq': 75,
             'n_scored': 225,
             'overall_acc': 0.6266666666666667},
 'timings': {'generation_sec': 1119.7211674520004,
             'prompt_build_sec': 0.020980745000997558,
             'retry_generation_sec': 20.915390042999206,
             'scoring_sec': 17.43489298500208}}
"""

In [ ]:
baseline2_private_result = run_baseline2_problem_set(
    problem_set=private_set,
    model_bundle=model_bundle,
    generation_config=baseline2_generation_config,
    retry_generation_config=baseline2_retry_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=False,
    output_jsonl_path=BASELINE2_DIR / "private_results.jsonl",
    debug_jsonl_path=BASELINE2_DIR / "private_debug.jsonl",
    submission_csv_path=BASELINE2_DIR / "submission.csv",
    report_json_path=BASELINE2_DIR / "private_report.json",
    show_progress=True,
)

pprint(baseline2_private_result.report)

In [ ]:
# At the end to download shit

!zip -r B2_results.zip /content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format

from google.colab import files
files.download('B2_results.zip')

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 1024,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'schema_accuracy': {'retry_used_accuracy': None,
                     'retry_used_n': 0,
                     'schema_invalid_accuracy': None,
                     'schema_invalid_n': 0,
                     'schema_valid_accuracy': None,
                     'schema_valid_n': 0},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 334.20872846600014,
             'prompt_build_sec': 0.07997702800003026,
             'retry_generation_sec': 279.99592230300004,
             'scoring_sec': 0.004742979999718955}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 4096,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 1024,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': None,
                     'retry_used_n': 0,
                     'sanitized_accuracy': None,
                     'sanitized_n': 0,
                     'schema_invalid_accuracy': None,
                     'schema_invalid_n': 0,
                     'schema_valid_accuracy': None,
                     'schema_valid_n': 0},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 1360.692972251,
             'prompt_build_sec': 0.08002269100006743,
             'retry_generation_sec': 79.66110282999989,
             'scoring_sec': 0.008488914999816188}}
"""

In [ ]:
"""
'generation_config': {'do_sample': True,
                       'max_new_tokens': 32768,
                       'min_p': 0.0,
                       'presence_penalty': 0.0,
                       'repetition_penalty': 1.0,
                       'temperature': 0.6,
                       'top_k': 20,
                       'top_p': 0.95},
 'output_jsonl_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/private_results.jsonl',
 'problem_set': {'n': 943,
                 'n_answered': 0,
                 'n_free_form': 643,
                 'n_mcq': 300,
                 'name': 'private'},
 'retry_generation_config': {'do_sample': False,
                             'max_new_tokens': 32768,
                             'min_p': 0.0,
                             'presence_penalty': 0.0,
                             'repetition_penalty': 1.0,
                             'temperature': 0.0,
                             'top_k': -1,
                             'top_p': 1.0},
 'schema_accuracy': {'retry_used_accuracy': None,
                     'retry_used_n': 0,
                     'sanitized_accuracy': None,
                     'sanitized_n': 0,
                     'schema_invalid_accuracy': None,
                     'schema_invalid_n': 0,
                     'schema_valid_accuracy': None,
                     'schema_valid_n': 0},
 'score_available': False,
 'submission_csv_path': '/content/CSE151B_Kaggle/CSE151B_Kaggle/results/baseline2_prompt_format/submission.csv',
 'summary': {'free_form_acc': None,
             'mcq_acc': None,
             'n_correct': 0,
             'n_free_form': 0,
             'n_mcq': 0,
             'n_scored': 0,
             'overall_acc': None},
 'timings': {'generation_sec': 5990.060841825001,
             'prompt_build_sec': 0.5626943770002981,
             'retry_generation_sec': 307.0900928789997,
             'scoring_sec': 0.005678710997017333}}
  """

In [ ]:
pd.DataFrame([
    baseline2_val_result.report["formatting"]
]).T.rename(columns={0: "value"})

In [ ]:
baseline2_val_result.report["formatting"]["schema_error_counts"]

In [ ]:
bad_schema_rows = [
    row for row in baseline2_val_result.scored_rows
    if not row.get("schema_valid")
]

len(bad_schema_rows), bad_schema_rows[:3]

In [ ]:
comparison_rows = []

if "baseline1_val_result" in globals():
    comparison_rows.append({
        "baseline": "baseline1",
        **baseline1_val_result.report["summary"],
    })

comparison_rows.append({
    "baseline": "baseline2",
    **baseline2_val_result.report["summary"],
    "schema_valid_rate": baseline2_val_result.report["formatting"]["schema_valid_rate"],
    "extractable_rate": baseline2_val_result.report["formatting"]["extractable_rate"],
    "retry_rate": baseline2_val_result.report["formatting"]["retry_rate"],
})

pd.DataFrame(comparison_rows)

# Baseline 3

In [ ]:
from baseline.baseline2_runner import run_baseline2_problem_set, run_baseline3_problem_set
from baseline.category_tagging import tag_problem_set_with_qwen, category_distribution
from baseline.experiments import append_comparison_row, build_comparison_row
from prompting.strategies import build_default_strategy_registry

BASELINE3_DIR = RESULTS_DIR / "baseline3_qwen_category_prompts"
BASELINE3_DIR.mkdir(parents=True, exist_ok=True)

PROMPT_STRATEGY_DIR = RESULTS_DIR / "prompt_strategy_experiments"
PROMPT_STRATEGY_DIR.mkdir(parents=True, exist_ok=True)

EXPERIMENT_COMPARISON_CSV = RESULTS_DIR / "experiment_comparison.csv"

BASELINE3_TAGGING_DIR = RESULTS_DIR / "baseline3_category_tagging"
BASELINE3_TAGGING_DIR.mkdir(parents=True, exist_ok=True)

baseline3_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=True,
)

baseline3_retry_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=False,
)

category_tag_generation_config = GenerationConfig(
    max_new_tokens=MAX_TOKENS,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=False,
)

print("Tagging output dir:", BASELINE3_TAGGING_DIR)

print("Baseline 3 output dir:", BASELINE3_DIR)
print("Comparison CSV:", EXPERIMENT_COMPARISON_CSV)

In [ ]:
strategy_registry = build_default_strategy_registry()
strategy_rows = []

for strategy_name in strategy_registry.names():
    strategy = strategy_registry.get(strategy_name)
    for route in strategy.routes:
        strategy_rows.append({
            "strategy": strategy.name,
            "label": strategy.label,
            "route": route.name,
            "answer_format": route.answer_format,
            "category": route.category,
            "template_name": route.template_name or route.name,
        })

strategy_df = pd.DataFrame(strategy_rows)
display(strategy_df)
print("Strategies:", strategy_registry.names())

## Question category tagging

In [ ]:
category_tag_generation_config.max_new_tokens

In [ ]:
BASELINE3_TAG_SMOKE_LIMIT = 8

tagged_val_smoke, val_tag_smoke_rows = tag_problem_set_with_qwen(
    problem_set=val_set,
    model_bundle=model_bundle,
    generation_config=category_tag_generation_config,
    batch_size=BATCH_SIZE,
    output_jsonl_path=BASELINE3_TAGGING_DIR / "val_category_tags_smoke.jsonl",
    output_one_hot_csv_path=BASELINE3_TAGGING_DIR / "val_category_one_hot_smoke.csv",
    limit=BASELINE3_TAG_SMOKE_LIMIT,
    show_progress=True,
)

smoke_df = pd.DataFrame([{
    "id": row["id"],
    "primary_category": row["primary_category"],
    "qwen_categories": row["qwen_categories"],
    "parse_ok": row["category_tag_parse_ok"],
    "source": row.get("category_tag_source"),
    "raw_head": row["category_tag_raw_output"][:180],
    "raw_tail": row["category_tag_raw_output"][-240:],
} for row in val_tag_smoke_rows])

display(smoke_df)

In [ ]:
for row in val_tag_smoke_rows:
    print("=" * 100)
    print("id:", row["id"])
    print("primary_category:", row["primary_category"])
    print("parse_ok:", row["category_tag_parse_ok"])
    print("source:", row.get("category_tag_source"))
    print(row["category_tag_raw_output"][:2000])

In [ ]:
category_cols = [
    "statistics_probability",
    "calculus",
    "geometry_trig",
    "linear_algebra",
    "discrete_algorithm",
    "arithmetic_algebra",
    "applied_word_problem",
    "general_math",
]

one_hot_smoke = pd.read_csv(BASELINE3_TAGGING_DIR / "val_category_one_hot_smoke.csv")
one_hot_smoke["num_categories"] = one_hot_smoke[category_cols].sum(axis=1)

display(one_hot_smoke[["id", "primary_category", "category_tag_source", "num_categories"]])
print(one_hot_smoke["num_categories"].value_counts())
print(one_hot_smoke["primary_category"].value_counts())

In [ ]:
tagged_public_set, public_category_rows = tag_problem_set_with_qwen(
    problem_set=public_set,
    model_bundle=model_bundle,
    generation_config=category_tag_generation_config,
    batch_size=BATCH_SIZE,
    output_jsonl_path=BASELINE3_TAGGING_DIR / "public_category_tags.jsonl",
    output_one_hot_csv_path=BASELINE3_TAGGING_DIR / "public_category_one_hot.csv",
    existing_tags_path=BASELINE3_TAGGING_DIR / "public_category_tags.jsonl",
    limit=None,
    show_progress=True,
)

print("Tagged public rows:", len(public_category_rows))
print(category_distribution(public_category_rows))

In [ ]:
public_one_hot = pd.read_csv(BASELINE3_TAGGING_DIR / "public_category_one_hot.csv")
public_one_hot["num_categories"] = public_one_hot[category_cols].sum(axis=1)

print("One-hot category count check:")
print(public_one_hot["num_categories"].value_counts())

print("\nPrimary category distribution:")
display(public_one_hot["primary_category"].value_counts().to_frame("count"))

print("\nTag source distribution:")
display(public_one_hot["category_tag_source"].value_counts().to_frame("count"))

In [ ]:
public_tags_df = pd.DataFrame(public_category_rows)

for category in category_cols:
    sample = public_tags_df[public_tags_df["primary_category"] == category].head(5)
    print("=" * 100)
    print("CATEGORY:", category, "n =", len(public_tags_df[public_tags_df["primary_category"] == category]))
    for _, row in sample.iterrows():
        print("-" * 80)
        print("id:", row["id"])
        print(str(row["question"])[:500])

In [ ]:
tagged_private_set, private_category_rows = tag_problem_set_with_qwen(
    problem_set=private_set,
    model_bundle=model_bundle,
    generation_config=category_tag_generation_config,
    batch_size=BATCH_SIZE,
    output_jsonl_path=BASELINE3_TAGGING_DIR / "private_category_tags.jsonl",
    output_one_hot_csv_path=BASELINE3_TAGGING_DIR / "private_category_one_hot.csv",
    existing_tags_path=BASELINE3_TAGGING_DIR / "private_category_tags.jsonl",
    limit=None,
    show_progress=True,
)

print("Tagged private rows:", len(private_category_rows))
print(category_distribution(private_category_rows))

In [ ]:
private_one_hot = pd.read_csv(BASELINE3_TAGGING_DIR / "private_category_one_hot.csv")
private_one_hot["num_categories"] = private_one_hot[category_cols].sum(axis=1)

print(private_one_hot["num_categories"].value_counts())
display(private_one_hot["primary_category"].value_counts().to_frame("count"))
display(private_one_hot["category_tag_source"].value_counts().to_frame("count"))

## Running category-specific experiments

In [ ]:
import importlib

import baseline.rule_guidance
import baseline.category_rules
import baseline.category_harnesses
import baseline.category_work.calculus
import baseline.category_work.general_math
import baseline.category_work.linear_algebra
import baseline.category_work.discrete_algorithm
import baseline.baseline3_prompts
import prompting.strategies
import prompting.templates
import prompting.prompt_chain
import baseline.category_experiments

from baseline.category_harnesses import (
    CategoryHarness,
    HarnessCheck,
    register_harness,
    get_harness,
)

from baseline.rule_guidance import register_default_linear_discrete_guidance, register_default_ved_guidance
from baseline.category_rules import (
    prepare_and_save_rule_annotations,
    apply_rule_annotations,
    rule_distribution,

)
from baseline.category_experiments import (
    load_tagged_problem_set,
    category_counts,
    run_category_strategy_ablation,
    ablation_plan_frame,
    ablation_summary_frame,
    ablation_wrong_rows_frame,
    print_ablation_wrong_rows,
    run_rule_strategy_ablation,
    category_summary_frame_rows,
)

CATEGORY_EXPERIMENT_DIR = RESULTS_DIR / "baseline3_category_experiments"
CATEGORY_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)

CATEGORY_EXPERIMENT_COMPARISON_CSV = RESULTS_DIR / "baseline3_category_experiment_comparison.csv"

registered_category_work = {}

for module in [
    baseline.category_work.calculus,
    baseline.category_work.general_math,
    baseline.category_work.linear_algebra,
    baseline.category_work.discrete_algorithm,
]:
    info = module.register_all()
    registered_category_work[info["category"]] = info

register_default_linear_discrete_guidance()
register_default_ved_guidance()

registered_category_work

In [ ]:
PUBLIC_DATA_PATH = "data/public.jsonl"

PUBLIC_CATEGORY_TAGS_PATH = (
    RESULTS_DIR
    / "baseline3_category_tagging"
    / "public_category_tags.jsonl"
)

tagged_public_set = load_tagged_problem_set(
    data_jsonl_path=PUBLIC_DATA_PATH,
    tags_jsonl_path=PUBLIC_CATEGORY_TAGS_PATH,
    name="public_tagged",
)

print(tagged_public_set.summary())
print(category_counts(tagged_public_set))

category_df = pd.DataFrame(category_summary_frame_rows(tagged_public_set)).sort_values(
    "n",
    ascending=False,
)

display(category_df)

In [ ]:
RULE_ANNOTATION_DIR = RESULTS_DIR / "baseline3_rule_annotations"
RULE_ANNOTATION_DIR.mkdir(parents=True, exist_ok=True)

RULE_CATEGORIES = [
    "calculus",
    "general_math",
    "linear_algebra",
    "discrete_algorithm",
]

tagged_public_with_rules = prepare_and_save_rule_annotations(
    problem_set=tagged_public_set,
    categories=RULE_CATEGORIES,
    output_jsonl_path=RULE_ANNOTATION_DIR / "public_category_rules.jsonl",
    output_one_hot_csv_path=RULE_ANNOTATION_DIR / "public_category_rule_one_hot.csv",
    name="public_tagged_with_category_rules",
)

print(tagged_public_with_rules.summary())
print(rule_distribution([
    r for r in tagged_public_with_rules.records
    if r.get("primary_category") in RULE_CATEGORIES
]))

In [ ]:
rule_rows = []

for record in tagged_public_with_rules.records:
    if record.get("primary_category") not in RULE_CATEGORIES:
        continue

    rule_rows.append({
        "id": record.get("id"),
        "category": record.get("primary_category"),
        "is_mcq": bool(record.get("options")),
        "derived_rules": record.get("derived_rules"),
        "question": str(record.get("question", ""))[:250],
    })

rule_df = pd.DataFrame(rule_rows)

display(rule_df.head(20))
display(rule_df.explode("derived_rules")["derived_rules"].value_counts().to_frame("count"))

In [ ]:
from prompting.prompt_chain import build_prompt_chain
from prompting.models import problem_from_record

sample_record = next(
    r for r in tagged_public_with_rules.records
    if r.get("primary_category") == "linear_algebra" and r.get("derived_rules")
)

chain = build_prompt_chain(strategy_name="baseline3_adaptive_rules")
spec = chain.build_spec(problem_from_record(sample_record))

print("id:", sample_record["id"])
print("rules:", sample_record["derived_rules"])
print("template:", spec.name)
print("=" * 100)
print(spec.to_messages()[0]["content"][:1200])
print("=" * 100)
print(spec.to_messages()[-1]["content"][:1800])

In [ ]:
ABLATION_CATEGORIES = [
    "calculus",
    "general_math",
    "linear_algebra",
    "discrete_algorithm",
]

ABLATION_STRATEGIES = [
    {"name": "baseline3", "categories": "all", "label": "category_guided_baseline"},
    {"name": "baseline3_adaptive_rules", "categories": "all", "label": "category_plus_derived_rules"},
]

plan_df, skipped_df = ablation_plan_frame(
    ABLATION_CATEGORIES,
    ABLATION_STRATEGIES,
)

print("Will run:")
display(plan_df)

print("Will skip:")
display(skipped_df)

In [ ]:
SMOKE_LIMIT_PER_COMBO = 8

smoke_ablation = run_category_strategy_ablation(
    problem_set=tagged_public_with_rules,
    categories=ABLATION_CATEGORIES,
    strategies=ABLATION_STRATEGIES,
    model_bundle=model_bundle,
    generation_config=baseline3_generation_config,
    retry_generation_config=baseline3_retry_generation_config,
    experiment_name="category_adaptive_rules_smoke",
    batch_size=BATCH_SIZE,
    limit_per_combo=SMOKE_LIMIT_PER_COMBO,
    score=True,
    output_dir=CATEGORY_EXPERIMENT_DIR,
    comparison_csv_path=None,
    show_progress=True,
)

smoke_summary_df = ablation_summary_frame(smoke_ablation)
display(smoke_summary_df.sort_values(["category", "strategy_name"]))
print(smoke_ablation["artifacts"])

In [ ]:
wrong_smoke_df = ablation_wrong_rows_frame(
    smoke_ablation,
    category="discrete_algorithm",
    strategy_name="baseline3_adaptive_rules",
    max_rows=10,
)

display(wrong_smoke_df[
    [
        "id",
        "is_mcq",
        "gold",
        "boxed_answer",
        "correct",
        "schema_valid",
        "question",
        "response_tail",
    ]
])

In [ ]:
print_ablation_wrong_rows(
    smoke_ablation,
    category="discrete_algorithm",
    strategy_name="baseline3_adaptive_rules",
    max_rows=5,
    max_question_chars=1200,
    max_response_chars=2500,
)

In [ ]:
full_ablation = run_category_strategy_ablation(
    problem_set=tagged_public_with_rules,
    categories=ABLATION_CATEGORIES,
    strategies=ABLATION_STRATEGIES,
    model_bundle=model_bundle,
    generation_config=baseline3_generation_config,
    retry_generation_config=baseline3_retry_generation_config,
    experiment_name="category_adaptive_rules_full",
    batch_size=BATCH_SIZE,
    limit_per_combo=None,
    score=True,
    output_dir=CATEGORY_EXPERIMENT_DIR,
    comparison_csv_path=None,
    show_progress=True,
)

full_summary_df = ablation_summary_frame(full_ablation)
display(
    full_summary_df.sort_values(
        ["share_of_ablation_errors", "category", "strategy_name"],
        ascending=[False, True, True],
    )
)

print(full_ablation["artifacts"])

In [ ]:
wrong_df = ablation_wrong_rows_frame(
    full_ablation,
    category="discrete_algorithm",
    strategy_name="baseline3_adaptive_rules",
    max_rows=30,
    max_question_chars=1000,
    max_response_chars=2500,
)

display(wrong_df[
    [
        "id",
        "is_mcq",
        "gold",
        "boxed_answer",
        "correct",
        "schema_valid",
        "harness_pass_rate",
        "question",
        "response_tail",
    ]
])

In [ ]:
print_ablation_wrong_rows(
    full_ablation,
    category="discrete_algorithm",
    strategy_name="baseline3_adaptive_rules",
    max_rows=30,
    max_question_chars=1200,
    max_response_chars=2500,
)

Each category file under category_work should contain:

1. category name
2. strategy names the teammate is testing
3. harness definition
4. derived rules
5. notes on where prompt templates / strategy registry entries live

## Running full problem set

In [ ]:
baseline3_prompt_chain = build_prompt_chain(strategy_name="baseline3")
baseline3_prompt_rows = []

for problem in baseline3_tagged_val_smoke.problems()[:5]:
    spec = baseline3_prompt_chain.build_spec(problem)
    baseline3_prompt_rows.append({
        "id": problem.id,
        "template_name": spec.name,
        "route_name": spec.metadata.get("route_name"),
        "category": spec.metadata.get("category"),
        "qwen_categories": spec.metadata.get("qwen_categories"),
        "answer_format": spec.metadata.get("answer_format"),
    })

pd.DataFrame(baseline3_prompt_rows)

In [ ]:
# Baseline 3 validation smoke run using Qwen category tags.

baseline3_val_smoke_result = run_baseline3_problem_set(
    problem_set=baseline3_tagged_val_smoke,
    model_bundle=model_bundle,
    generation_config=baseline3_generation_config,
    retry_generation_config=baseline3_retry_generation_config,
    batch_size=BATCH_SIZE,
    limit=None,
    score=True,
    output_jsonl_path=BASELINE3_DIR / "val_smoke_results.jsonl",
    debug_jsonl_path=BASELINE3_DIR / "val_smoke_debug.jsonl",
    report_json_path=BASELINE3_DIR / "val_smoke_report.json",
    comparison_csv_path=EXPERIMENT_COMPARISON_CSV,
    experiment_name="baseline3_qwen_category_prompts_smoke",
    split_name="val_smoke",
    show_progress=True,
)

pprint({
    "summary": baseline3_val_smoke_result.report["summary"],
    "formatting": baseline3_val_smoke_result.report["formatting"],
    "category_counts": baseline3_val_smoke_result.report["category_counts"],
    "comparison_csv": str(EXPERIMENT_COMPARISON_CSV),
})

In [ ]:
def tag_split_for_baseline3(problem_set, split_name, limit=None, reuse_existing=True):
    tag_path = BASELINE3_DIR / f"{split_name}_category_tags.jsonl"
    existing_tags_path = tag_path if reuse_existing and tag_path.exists() else None

    tagged_set, tag_rows = tag_problem_set_with_qwen(
        problem_set=problem_set,
        model_bundle=model_bundle,
        generation_config=category_tag_generation_config,
        batch_size=BATCH_SIZE,
        output_jsonl_path=tag_path,
        existing_tags_path=existing_tags_path,
        limit=limit,
        show_progress=True,
    )

    print(split_name, "tag rows:", len(tag_rows), "path:", tag_path)
    return tagged_set, tag_rows, tag_path


def run_baseline3_split(problem_set, split_name, score, submission=False, limit=None):
    tagged_set, tag_rows, tag_path = tag_split_for_baseline3(
        problem_set=problem_set,
        split_name=split_name,
        limit=limit,
        reuse_existing=True,
    )

    report_path = BASELINE3_DIR / f"{split_name}_report.json"
    result = run_baseline3_problem_set(
        problem_set=tagged_set,
        model_bundle=model_bundle,
        generation_config=baseline3_generation_config,
        retry_generation_config=baseline3_retry_generation_config,
        batch_size=BATCH_SIZE,
        limit=None,
        score=score,
        output_jsonl_path=BASELINE3_DIR / f"{split_name}_results.jsonl",
        debug_jsonl_path=BASELINE3_DIR / f"{split_name}_debug.jsonl",
        submission_csv_path=BASELINE3_DIR / "submission.csv" if submission else None,
        report_json_path=report_path,
        comparison_csv_path=EXPERIMENT_COMPARISON_CSV,
        experiment_name="baseline3_qwen_category_prompts",
        split_name=split_name,
        show_progress=True,
    )

    result.report["category_tags_path"] = str(tag_path)
    result.report["category_tag_count"] = len(tag_rows)
    write_report(result.report, report_path)
    return result

In [ ]:
baseline3_val_result = run_baseline3_split(
    problem_set=val_set,
    split_name="val",
    score=True,
    submission=False,
    limit=None,
)
pprint(baseline3_val_result.report)

## Due Today: Ved-Only Category Run

This section runs only `calculus` and `general_math`. It is independent of the older smoke cells and writes honest validation metrics plus failure-analysis artifacts.


In [ ]:
# Due-today focused run: only calculus and general_math.
# Run this after setup, dataset loading, model loading, and the Baseline 3 config cell.

from pathlib import Path
from pprint import pprint
import pandas as pd

from baseline.category_experiments import run_category_strategy_grid, category_counts
from baseline.category_rules import prepare_and_save_rule_annotations
from baseline.rule_guidance import register_default_ved_guidance
from baseline.ved_category_analysis import build_failure_analysis_rows, save_failure_analysis
from run_ved_category_experiments import strategy_grid, write_experiment_log, save_grid_failure_analysis

import baseline.category_work.calculus as calculus_work
import baseline.category_work.general_math as general_math_work

calculus_work.register_all()
general_math_work.register_all()
register_default_ved_guidance()

VED_CATEGORIES = ("calculus", "general_math")
VED_EXPERIMENT_DIR = RESULTS_DIR / "ved_category_experiments_due_today"
VED_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
VED_COMPARISON_CSV = RESULTS_DIR / "ved_category_experiment_comparison_due_today.csv"

# Set to a small number like 3 for a quick connectivity test. Set to None for all validation rows in these categories.
VED_LIMIT_PER_CATEGORY = 3

# Use a smaller token budget for the category experiments to avoid long hangs.
ved_generation_config = GenerationConfig(
    max_new_tokens=384,
    temperature=0.0,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=False,
)

ved_retry_generation_config = GenerationConfig(
    max_new_tokens=96,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=False,
)

ved_tag_generation_config = GenerationConfig(
    max_new_tokens=64,
    temperature=0.0,
    top_p=1.0,
    top_k=-1,
    min_p=0.0,
    repetition_penalty=1.0,
    presence_penalty=0.0,
    do_sample=False,
)

print("Ved output dir:", VED_EXPERIMENT_DIR)
print("Ved comparison CSV:", VED_COMPARISON_CSV)
print("Ved limit per category:", VED_LIMIT_PER_CATEGORY)



In [ ]:
# Tag validation with Qwen, reusing saved tags if they already exist.
# If baseline3_val_result already ran, this should reuse the saved val_category_tags.jsonl.

ved_tag_path = BASELINE3_DIR / "val_category_tags.jsonl"
existing_ved_tags = ved_tag_path if ved_tag_path.exists() else None

ved_tagged_val_set, ved_tag_rows = tag_problem_set_with_qwen(
    problem_set=val_set,
    model_bundle=model_bundle,
    generation_config=ved_tag_generation_config,
    batch_size=1,
    output_jsonl_path=ved_tag_path,
    existing_tags_path=existing_ved_tags,
    limit=None,
    show_progress=True,
)

ved_tagged_val_with_rules = prepare_and_save_rule_annotations(
    problem_set=ved_tagged_val_set,
    categories=VED_CATEGORIES,
    output_jsonl_path=VED_EXPERIMENT_DIR / "val_ved_rule_annotations.jsonl",
    output_one_hot_csv_path=VED_EXPERIMENT_DIR / "val_ved_rule_one_hot.csv",
    name="val_tagged_ved_rules",
)

print("Tag rows:", len(ved_tag_rows))
print("Category counts:")
print(category_counts(ved_tagged_val_with_rules))
print("Strategies:")
pprint(strategy_grid())



In [ ]:
# Run honest validation experiments for only calculus and general_math.
# This does not use labels to choose predictions; labels are used only for scoring.

ved_results_by_category = {}

for category, strategies in strategy_grid().items():
    ved_results_by_category[category] = run_category_strategy_grid(
        problem_set=ved_tagged_val_with_rules,
        category=category,
        strategy_names=strategies,
        model_bundle=model_bundle,
        generation_config=ved_generation_config,
        retry_generation_config=ved_retry_generation_config,
        experiment_name=f"val_{category}_ved_due_today",
        batch_size=1,
        limit=VED_LIMIT_PER_CATEGORY,
        score=True,
        output_dir=VED_EXPERIMENT_DIR,
        comparison_csv_path=VED_COMPARISON_CSV,
        show_progress=True,
    )

ved_failure_artifacts = save_grid_failure_analysis(
    ved_results_by_category,
    output_dir=VED_EXPERIMENT_DIR,
)
ved_experiment_log_path = write_experiment_log(
    ved_results_by_category,
    output_dir=VED_EXPERIMENT_DIR,
)

ved_log_df = pd.read_csv(ved_experiment_log_path).sort_values(
    ["category", "accuracy_delta_vs_baseline3"],
    ascending=[True, False],
)

print("Ved experiment log:", ved_experiment_log_path)
print("Failure artifacts written under:", VED_EXPERIMENT_DIR)
display(ved_log_df)



In [ ]:
# Review every wrong prediction for the best strategy per category from the experiment log.

best_strategy_by_category = {}
for category, group in ved_log_df.groupby("category"):
    scored = group.dropna(subset=["accuracy_after"])
    if len(scored):
        best_strategy_by_category[category] = scored.sort_values("accuracy_after", ascending=False).iloc[0]["strategy_name"]

print("Best strategy by category:")
pprint(best_strategy_by_category)

for category, strategy_name in best_strategy_by_category.items():
    failure_csv = (
        VED_EXPERIMENT_DIR
        / category
        / "failure_analysis"
        / strategy_name
        / f"{category}_{strategy_name}_failure_analysis.csv"
    )
    print("= " * 50)
    print(category, strategy_name, failure_csv)
    if failure_csv.exists():
        failure_df = pd.read_csv(failure_csv)
        if "no_failures" in failure_df.columns:
            print("No failures for this strategy/category.")
        else:
            display(failure_df[[
                "id",
                "ground_truth_answer",
                "model_answer",
                "prompt_template_used",
                "strategy_used",
                "root_cause_classification",
                "why_answer_is_wrong",
                "question",
            ]])
    else:
        print("No failure CSV found for this category/strategy.")



## Ved category validation experiments

Run this section after the Baseline 3 validation run. It evaluates only `calculus` and `general_math`, writes strategy comparison metrics, and saves every incorrect prediction with root-cause labels.


In [ ]:
from run_ved_category_experiments import (
    strategy_grid,
    write_experiment_log,
    save_grid_failure_analysis,
)
from baseline.category_experiments import run_category_strategy_grid, category_counts
from baseline.category_rules import prepare_and_save_rule_annotations

VED_CATEGORIES = ("calculus", "general_math")
VED_EXPERIMENT_DIR = RESULTS_DIR / "ved_category_experiments"
VED_EXPERIMENT_DIR.mkdir(parents=True, exist_ok=True)
VED_COMPARISON_CSV = RESULTS_DIR / "ved_category_experiment_comparison.csv"

# Reuse the validation category tags when present; otherwise this tags val with Qwen.
tagged_ved_val_set, _, _ = tag_split_for_baseline3(
    problem_set=val_set,
    split_name="val",
    limit=None,
    reuse_existing=True,
)

tagged_ved_val_with_rules = prepare_and_save_rule_annotations(
    problem_set=tagged_ved_val_set,
    categories=VED_CATEGORIES,
    output_jsonl_path=VED_EXPERIMENT_DIR / "val_ved_rule_annotations.jsonl",
    output_one_hot_csv_path=VED_EXPERIMENT_DIR / "val_ved_rule_one_hot.csv",
    name="val_tagged_ved_rules",
)

print("Ved validation category counts:")
print(category_counts(tagged_ved_val_with_rules))
print("Ved strategy grid:")
pprint(strategy_grid())



In [ ]:
ved_results_by_category = {}

for category, strategies in strategy_grid().items():
    ved_results_by_category[category] = run_category_strategy_grid(
        problem_set=tagged_ved_val_with_rules,
        category=category,
        strategy_names=strategies,
        model_bundle=model_bundle,
        generation_config=baseline3_generation_config,
        retry_generation_config=baseline3_retry_generation_config,
        experiment_name=f"val_{category}_ved_strategy_grid",
        batch_size=BATCH_SIZE,
        limit=None,
        score=True,
        output_dir=VED_EXPERIMENT_DIR,
        comparison_csv_path=VED_COMPARISON_CSV,
        show_progress=True,
    )

ved_failure_artifacts = save_grid_failure_analysis(
    ved_results_by_category,
    output_dir=VED_EXPERIMENT_DIR,
)
ved_experiment_log_path = write_experiment_log(
    ved_results_by_category,
    output_dir=VED_EXPERIMENT_DIR,
)

print("Ved experiment log:", ved_experiment_log_path)
print("Ved comparison CSV:", VED_COMPARISON_CSV)
print("Failure artifact keys:")
pprint(sorted(ved_failure_artifacts))

display(pd.read_csv(ved_experiment_log_path).sort_values(
    ["category", "accuracy_delta_vs_baseline3"],
    ascending=[True, False],
))



In [ ]:
# Inspect the full failure table for the best current Ved strategy in each category.
# Change the strategy names below after reviewing ved_experiment_log.csv.
VED_FAILURE_REVIEW = {
    "calculus": "calculus_v1_structured",
    "general_math": "general_math_v1_structured",
}

for category, strategy_name in VED_FAILURE_REVIEW.items():
    failure_csv = (
        VED_EXPERIMENT_DIR
        / category
        / "failure_analysis"
        / strategy_name
        / f"{category}_{strategy_name}_failure_analysis.csv"
    )
    print("= " * 50)
    print(category, strategy_name, failure_csv)
    if failure_csv.exists():
        failure_df = pd.read_csv(failure_csv)
        display(failure_df[[
            "id",
            "ground_truth_answer",
            "model_answer",
            "prompt_template_used",
            "strategy_used",
            "root_cause_classification",
            "why_answer_is_wrong",
            "question",
        ]])
    else:
        print("No failure CSV found for this category/strategy.")



In [ ]:
# Full Baseline 3 private run. This writes the Kaggle-compatible submission.csv.

RUN_BASELINE3_PRIVATE = False

if RUN_BASELINE3_PRIVATE:
    baseline3_private_result = run_baseline3_split(
        problem_set=private_set,
        split_name="private",
        score=False,
        submission=True,
        limit=None,
    )
    pprint(baseline3_private_result.report)
else:
    print("Set RUN_BASELINE3_PRIVATE = True to generate the Baseline 3 private submission.")

In [ ]:
def run_prompt_chain_strategy_suite(problem_set, split_name="val_strategy_smoke", limit=8):
    strategies = ["baseline", "baseline2", "baseline3"]
    results = {}
    tagged_set = None

    for strategy_name in strategies:
        if strategy_name == "baseline3":
            if tagged_set is None:
                tagged_set, _, _ = tag_split_for_baseline3(
                    problem_set=problem_set,
                    split_name=split_name,
                    limit=limit,
                    reuse_existing=True,
                )
            run_set = tagged_set
            run_limit = None
            report_label = "baseline3_qwen_category_prompts"
        else:
            run_set = problem_set
            run_limit = limit
            report_label = {
                "baseline": "baseline_weakest_prompt_chain",
                "baseline2": "baseline2_prompt_format",
            }[strategy_name]

        strategy_dir = PROMPT_STRATEGY_DIR / strategy_name
        strategy_dir.mkdir(parents=True, exist_ok=True)

        result = run_baseline2_problem_set(
            problem_set=run_set,
            model_bundle=model_bundle,
            generation_config=baseline3_generation_config,
            retry_generation_config=baseline3_retry_generation_config,
            batch_size=BATCH_SIZE,
            limit=run_limit,
            score=(split_name != "private"),
            strategy_name=strategy_name,
            report_label=report_label,
            output_jsonl_path=strategy_dir / f"{split_name}_results.jsonl",
            debug_jsonl_path=strategy_dir / f"{split_name}_debug.jsonl",
            report_json_path=strategy_dir / f"{split_name}_report.json",
            comparison_csv_path=EXPERIMENT_COMPARISON_CSV,
            experiment_name=f"prompt_chain_{strategy_name}",
            split_name=split_name,
            show_progress=True,
        )
        results[strategy_name] = result

    return results


RUN_PROMPT_STRATEGY_SUITE = False

if RUN_PROMPT_STRATEGY_SUITE:
    prompt_strategy_results = run_prompt_chain_strategy_suite(
        problem_set=val_set,
        split_name="val_strategy_smoke",
        limit=8,
    )
    display(pd.DataFrame([{
        "strategy": name,
        **result.report["summary"],
        "schema_valid_rate": result.report["formatting"]["schema_valid_rate"],
        "extractable_rate": result.report["formatting"]["extractable_rate"],
        "retry_rate": result.report["formatting"]["retry_rate"],
    } for name, result in prompt_strategy_results.items()]))
else:
    print("Set RUN_PROMPT_STRATEGY_SUITE = True to compare prompt-chain strategies.")

In [ ]:
if EXPERIMENT_COMPARISON_CSV.exists():
    comparison_df = pd.read_csv(EXPERIMENT_COMPARISON_CSV)
    display(comparison_df.tail(10))
else:
    print("No comparison CSV yet.")

# Baseline 4: Supervised Fine-Tuning (SFT)

## Post-SFT Tuning

# Baseline 5: RL